In [2]:
#imports
import os
from pathlib import Path
from PIL import Image,ImageDraw,ImageFont
import cv2
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import random

# Disable LaTeX engine
matplotlib.rcParams['text.usetex'] = False

# (Optional but recommended)
matplotlib.rcParams['mathtext.default'] = 'regular'
print("Imported Successfully")

Imported Successfully


In [13]:
# -----------------------------
# CONFIG
# -----------------------------
IMAGE_DIR = "dataset/faces_compressed"
ORIGINAL_DIR = "dataset/faces"

# ASCII charset (dark → light)
#char_set = """$@B%8&WM#*oahkbdpqwmZO0QLCJUYXzcvunxrjft/\|()1{}[]?-_+~<>i!lI;:,"^`'."""
char_set = "@#W$9876543210?!abc;:+=-,._"  # heavy → light
#char_set = """$@B%8&WM#*oahkbdpqwmZO0QLCJUYXzcvunxrjft/\|()1{}[]?-_+~<>i!lI;:,"^`'."""
#char_set = "█▓▒░"
#char_set = "@. "
char_set = " .:coPO?@■"[::-1]
# Index 0: Vertical, 1: Horizontal, 2: Diagonal Forward, 3: Diagonal Backward
edge_chars = ['|', '-', '/', '\\']

In [4]:
"""#----------------------------
#ASCII Shader
#-----------------------------
def image_to_ascii(image, width=128):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    h, w = gray.shape
    aspect_ratio = h / w
    height = int(width * aspect_ratio)

    resized = cv2.resize(gray, (width, height))

    ascii_img = []
    for row in resized:
        line = ""
        for pixel in row:
            idx = int(pixel / 255 * (len(char_set) - 1))
            line += char_set[idx]
        ascii_img.append(line)

    return ascii_img"""

'#----------------------------\n#ASCII Shader\n#-----------------------------\ndef image_to_ascii(image, width=128):\n    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)\n\n    h, w = gray.shape\n    aspect_ratio = h / w\n    height = int(width * aspect_ratio)\n\n    resized = cv2.resize(gray, (width, height))\n\n    ascii_img = []\n    for row in resized:\n        line = ""\n        for pixel in row:\n            idx = int(pixel / 255 * (len(char_set) - 1))\n            line += char_set[idx]\n        ascii_img.append(line)\n\n    return ascii_img'

In [5]:
# ==========================================
# ASCII SHADER FUNCTIONS
# ==========================================

# 1. BASE FILL (Luminance)
def get_luminance_fill(gray_img, grid_size):
    resized = cv2.resize(gray_img, (grid_size, grid_size), interpolation=cv2.INTER_AREA)
    
    grid = []
    for row in resized:
        line = []
        for pixel in row:
            idx = int((pixel / 255.0) * (len(char_set) - 1))
            line.append(char_set[idx])
        grid.append(line)
    return grid

# 2. EDGE MASK (Difference of Gaussians)
def get_difference_of_gaussians(gray_img, sigma, sigma_scale, tau, threshold):
    img_float = gray_img.astype(np.float32) / 255.0
    
    blur1 = cv2.GaussianBlur(img_float, (0, 0), sigmaX=sigma)
    blur2 = cv2.GaussianBlur(img_float, (0, 0), sigmaX=sigma * sigma_scale)
    
    dog = blur1 - (tau * blur2)
    dog_mask = (dog >= threshold).astype(np.uint8)
    return dog_mask

# 3. EDGE ANGLES (Sobel Filter)
def get_sobel_angles(gray_img):
    sobelx = cv2.Sobel(gray_img, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray_img, cv2.CV_64F, 0, 1, ksize=3)
    
    theta = np.arctan2(sobely, sobelx)
    angles_deg = np.degrees(theta) % 180
    
    direction_map = np.zeros_like(angles_deg, dtype=np.uint8)
    
    direction_map[(angles_deg >= 157.5) | (angles_deg < 22.5)] = 0
    direction_map[(angles_deg >= 22.5) & (angles_deg < 67.5)] = 3
    direction_map[(angles_deg >= 67.5) & (angles_deg < 112.5)] = 1
    direction_map[(angles_deg >= 112.5) & (angles_deg < 157.5)] = 2
    
    return direction_map

# 4. COMPUTE SHADER (Block Histogram)
def compute_block_edges(dog_mask, angle_map, grid_size, edge_threshold, strict_logic):
    h, w = dog_mask.shape
    block_size = h // grid_size  
    
    edge_grid = [['' for _ in range(grid_size)] for _ in range(grid_size)]
    
    for y in range(grid_size):
        for x in range(grid_size):
            y_start, y_end = y * block_size, (y + 1) * block_size
            x_start, x_end = x * block_size, (x + 1) * block_size
            
            dog_block = dog_mask[y_start:y_end, x_start:x_end]
            angle_block = angle_map[y_start:y_end, x_start:x_end]
            
            valid_edges = angle_block[dog_block == 1]
            
            if len(valid_edges) > 0:
                counts = np.bincount(valid_edges, minlength=4)
                dominant_direction = np.argmax(counts)
                
                if strict_logic:
                    # NEW WAY: The dominant angle *itself* must meet the threshold.
                    # Filters out blocks with noisy, scattered edges.
                    if counts[dominant_direction] >= edge_threshold:
                        edge_grid[y][x] = edge_chars[dominant_direction]
                else:
                    # OLD WAY: Total edges must meet threshold, regardless of direction.
                    # Causes the "pockmarked" look.
                    if len(valid_edges) >= edge_threshold:
                        edge_grid[y][x] = edge_chars[dominant_direction]
                
    return edge_grid

# 5. COMPOSITE
def composite_ascii(fill_grid, edge_grid):
    final_grid = []
    for r in range(len(fill_grid)):
        line = ""
        for c in range(len(fill_grid[r])):
            if edge_grid[r][c] != '':
                line += edge_grid[r][c]
            else:
                line += fill_grid[r][c]
        final_grid.append(line)
    return final_grid


# ==========================================
# MASTER PIPELINE ORCHESTRATOR
# ==========================================
def image_to_ascii(image, grid_size=128):
    # ----------------------------------------------------
    # 1. PIPELINE TOGGLES 
    # ----------------------------------------------------
    USE_BASE_FILL        = True  # Show underlying brightness characters
    USE_DOG_MASK         = True  # Use Gaussian difference to find actual lines
    USE_SOBEL_EDGES      = True  # Draw the \ | / - edge characters
    
    # NEW TOGGLES FOR NOISE REDUCTION:
    USE_BILATERAL_FILTER = True  # Pre-blurs flat surfaces (skin) but keeps edges sharp
    STRICT_EDGE_LOGIC    = True  # Forces edges to align in the same direction to be drawn

    # ----------------------------------------------------
    # 2. SHADER CONFIGURATIONS (Tweak these!)
    # ----------------------------------------------------
    # Bilateral Filter (If enabled)
    bilateral_d          = 9     # Radius of smoothing (higher = smoother skin)
    bilateral_sigma      = 75    # Strength of the color flattening
    
    # Difference of Gaussians (DoG) Edge Detection
    dog_sigma            = 0.87   # Base blur size. Lower = tighter, thinner lines
    dog_sigma_scale      = 1.6   # Secondary blur scale (standard is 1.6)
    dog_tau              = 0.98  # Edge falloff. Closer to 1.0 means thinner edges
    dog_threshold        = 0.09  # How much contrast makes an edge? Higher = ignores noise
    
    # Compute Shader (Block Logic)
    edge_threshold       = 8    # Out of 64 pixels, how many must be edges to draw? (0-64)
    # ----------------------------------------------------

    # Prep target resolution
    target_res = grid_size * 8
    gray_raw = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # --- PASS 1: Base Fill ---
    # We always use the raw, un-blurred image for the base fill so it retains lighting gradients
    if USE_BASE_FILL:
        gray_raw_scaled = cv2.resize(gray_raw, (target_res, target_res), interpolation=cv2.INTER_AREA)
        fill_grid = get_luminance_fill(gray_raw_scaled, grid_size)
    else:
        fill_grid = [[' ' for _ in range(grid_size)] for _ in range(grid_size)]

    # Early exit if we aren't drawing edges
    if not USE_SOBEL_EDGES:
        return ["".join(row) for row in fill_grid]

    # --- PASS 2: Pre-Processing for Edges ---
    if USE_BILATERAL_FILTER:
        gray_processed = cv2.bilateralFilter(gray_raw, d=bilateral_d, sigmaColor=bilateral_sigma, sigmaSpace=bilateral_sigma)
    else:
        gray_processed = gray_raw
        
    gray_scaled = cv2.resize(gray_processed, (target_res, target_res), interpolation=cv2.INTER_AREA)

    # --- PASS 3: Difference of Gaussians ---
    if USE_DOG_MASK:
        dog_mask = get_difference_of_gaussians(
            gray_scaled, 
            sigma=dog_sigma, 
            sigma_scale=dog_sigma_scale, 
            tau=dog_tau, 
            threshold=dog_threshold
        )
    else:
        dog_mask = np.ones_like(gray_scaled, dtype=np.uint8)

    # --- PASS 4: Sobel Filter ---
    angle_map = get_sobel_angles(gray_scaled)

    # --- PASS 5: Compute Block Edges ---
    edge_grid = compute_block_edges(
        dog_mask, 
        angle_map, 
        grid_size, 
        edge_threshold=edge_threshold, 
        strict_logic=STRICT_EDGE_LOGIC
    )

    # --- PASS 6: Composite ---
    final_ascii = composite_ascii(fill_grid, edge_grid)

    return final_ascii

In [29]:
def ascii_to_image(
    ascii_img,
    char_width=8,
    char_height=8,
    bg_color=(51,0,51),
    font_color=(255,139,50)
):
    rows = len(ascii_img)
    cols = len(ascii_img[0])

    img_width = cols * char_width
    img_height = rows * char_height

    img = Image.new("RGB", (img_width, img_height), color=bg_color)
    draw = ImageDraw.Draw(img)

    font = ImageFont.truetype("Courier New.ttf",9)

    for y, line in enumerate(ascii_img):
        for x, char in enumerate(line):
            draw.text(
                (x * char_width, y * char_height),
                char,
                font=font,
                fill=font_color
            )

    return img

In [27]:
# -----------------------------
# Load random image
# -----------------------------
files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")]
random_file = random.choice(files)

img_path = os.path.join(IMAGE_DIR, random_file)
original_path = os.path.join(ORIGINAL_DIR, random_file)

img = cv2.imread(img_path)
org = cv2.imread(original_path)

if img is None or org is None:
    raise ValueError("Failed to load image")

img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
org_rgb = cv2.cvtColor(org, cv2.COLOR_BGR2RGB)

# -----------------------------
# IMPORTANT: match ASCII width to image width
# -----------------------------
# generate ASCII
ascii_img = image_to_ascii(img)

# convert ASCII → image
ascii_render = ascii_to_image(ascii_img, char_width=8, char_height=8)

# convert for matplotlib
ascii_render_np = np.array(ascii_render)
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# ==================================================
# SAVE HIGH-RESOLUTION PNG
# ==================================================
save_path = "high_res_ascii_render.png"
ascii_render.save(save_path, "PNG")
print(f"✅ Successfully saved exact {ascii_render.width}x{ascii_render.height} render to: {save_path}")

# original
axes[0].imshow(org_rgb)
axes[0].set_title("Original")
axes[0].axis("off")
axes[0].set_aspect('equal')

# ASCII render
axes[1].imshow(ascii_render_np)
axes[1].set_title("ASCII Render")
axes[1].axis("off")
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

TypeError: compute_block_edges() got multiple values for argument 'edge_threshold'

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
import random

# ==========================================
# GLOBAL STATE & CONFIG
# ==========================================
IMAGE_DIR = "dataset/faces_compressed"
ORIGINAL_DIR = "dataset/faces"
char_set = " .:coPO?@■"
edge_chars = ['|', '-', '/', '\\']

current_img_raw = None
current_org_rgb = None

# ==========================================
# CORE SHADER PIPELINE
# ==========================================
def get_luminance_fill(gray_img, grid_size):
    resized = cv2.resize(gray_img, (grid_size, grid_size), interpolation=cv2.INTER_AREA)
    grid = []
    for row in resized:
        grid.append([char_set[int((pixel / 255.0) * (len(char_set) - 1))] for pixel in row])
    return grid

def get_difference_of_gaussians(gray_img, sigma, sigma_scale, tau, threshold):
    img_float = gray_img.astype(np.float32) / 255.0
    blur1 = cv2.GaussianBlur(img_float, (0, 0), sigmaX=sigma)
    blur2 = cv2.GaussianBlur(img_float, (0, 0), sigmaX=sigma * sigma_scale)
    dog = blur1 - (tau * blur2)
    # Returns binary map: 1 for edge, 0 for empty
    return (dog >= threshold).astype(np.uint8)

def get_sobel_angles(input_mask):
    # CRITICAL FIX: Run Sobel on float32 representation of the mask
    sobelx = cv2.Sobel(input_mask.astype(np.float32), cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(input_mask.astype(np.float32), cv2.CV_64F, 0, 1, ksize=3)
    
    # CRITICAL FIX: Calculate magnitude. If 0, it's a flat area, not an edge!
    magnitude = np.hypot(sobelx, sobely)
    theta = np.arctan2(sobely, sobelx)
    angles_deg = np.degrees(theta) % 180
    
    # 255 represents "NO EDGE" (Ignore flag)
    direction_map = np.full(angles_deg.shape, 255, dtype=np.uint8)
    
    # Only assign directions to pixels that actually have a gradient magnitude
    valid_mask = magnitude > 1e-5
    
    valid_dirs = np.zeros_like(angles_deg, dtype=np.uint8)
    valid_dirs[(angles_deg >= 157.5) | (angles_deg < 22.5)] = 0
    valid_dirs[(angles_deg >= 22.5) & (angles_deg < 67.5)] = 3
    valid_dirs[(angles_deg >= 67.5) & (angles_deg < 112.5)] = 1
    valid_dirs[(angles_deg >= 112.5) & (angles_deg < 157.5)] = 2
    
    direction_map[valid_mask] = valid_dirs[valid_mask]
    return direction_map

def compute_block_edges(angle_map, grid_size, edge_threshold, strict_logic):
    h, w = angle_map.shape
    block_size = h // grid_size  
    edge_grid = [['' for _ in range(grid_size)] for _ in range(grid_size)]
    
    for y in range(grid_size):
        for x in range(grid_size):
            y_start, y_end = y * block_size, (y + 1) * block_size
            x_start, x_end = x * block_size, (x + 1) * block_size
            
            angle_block = angle_map[y_start:y_end, x_start:x_end]
            
            # CRITICAL FIX: Ignore all '255' values (flat non-edge areas)
            valid_edges = angle_block[angle_block != 255]
            
            if len(valid_edges) > 0:
                counts = np.bincount(valid_edges, minlength=4)
                dominant_direction = np.argmax(counts)
                
                if strict_logic:
                    if counts[dominant_direction] >= edge_threshold:
                        edge_grid[y][x] = edge_chars[dominant_direction]
                else:
                    if len(valid_edges) >= edge_threshold:
                        edge_grid[y][x] = edge_chars[dominant_direction]
    return edge_grid

def composite_ascii(fill_grid, edge_grid):
    return ["".join([edge_grid[r][c] if edge_grid[r][c] != '' else fill_grid[r][c] for c in range(len(fill_grid[r]))]) for r in range(len(fill_grid))]

# ==========================================
# UI & INTERACTIVITY LOGIC
# ==========================================

out = widgets.Output()

def load_new_random_image(b=None):
    global current_img_raw, current_org_rgb
    files = [f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")]
    random_file = random.choice(files)
    
    img = cv2.imread(os.path.join(IMAGE_DIR, random_file))
    org = cv2.imread(os.path.join(ORIGINAL_DIR, random_file))
    
    current_img_raw = cv2.resize(img, (1024, 1024))
    current_org_rgb = cv2.cvtColor(cv2.resize(org, (1024, 1024)), cv2.COLOR_BGR2RGB)
    update_plot() 

def render_pipeline(
    view_debug, use_fill, use_dog, use_sobel, use_bilateral, strict_logic,
    bil_d, bil_sig, dog_sig, dog_scale, dog_tau, dog_thresh, edge_thresh
):
    if current_img_raw is None: return

    grid_size = 128
    target_res = grid_size * 8
    gray_raw = cv2.cvtColor(current_img_raw, cv2.COLOR_BGR2GRAY)
    
    # 1. Base Fill
    if use_fill:
        gray_raw_scaled = cv2.resize(gray_raw, (target_res, target_res), interpolation=cv2.INTER_AREA)
        fill_grid = get_luminance_fill(gray_raw_scaled, grid_size)
    else:
        fill_grid = [[' ' for _ in range(grid_size)] for _ in range(grid_size)]

    # 2. Bilateral Filter
    if use_bilateral:
        gray_processed = cv2.bilateralFilter(gray_raw, d=bil_d, sigmaColor=bil_sig, sigmaSpace=bil_sig)
    else:
        gray_processed = gray_raw
    gray_scaled = cv2.resize(gray_processed, (target_res, target_res), interpolation=cv2.INTER_AREA)

    # 3. Difference of Gaussians
    if use_dog:
        dog_mask = get_difference_of_gaussians(gray_scaled, dog_sig, dog_scale, dog_tau, dog_thresh)
    else:
        dog_mask = gray_scaled 
        
    # 4. Sobel Angles
    angle_map = get_sobel_angles(dog_mask if use_dog else gray_scaled)

    # 5. Compute & Composite
    if use_sobel:
        edge_grid = compute_block_edges(angle_map, grid_size, edge_thresh, strict_logic)
        ascii_img = composite_ascii(fill_grid, edge_grid)
    else:
        ascii_img = ["".join(row) for row in fill_grid]
        
    # ------------- RENDERING OUT -------------
    with out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        axes[0].imshow(current_org_rgb)
        axes[0].set_title("Original")
        axes[0].axis("off")
        
        if view_debug:
            axes[1].imshow(dog_mask, cmap='gray')
            axes[1].set_title("DEBUG: DoG Edge Mask")
        else:
            # Generate the PIL Image
            ascii_render = ascii_to_image(ascii_img, char_width=8, char_height=8)
            
            # --- NEW CODE: SAVE HIGH QUALITY ---
            save_path = "current_ascii_render.png"
            ascii_render.save(save_path, "PNG")
            # -----------------------------------

            axes[1].imshow(np.array(ascii_render))
            axes[1].set_title("ASCII Render (Saved to disk!)")
            
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()

# ==========================================
# WIDGET DEFINITIONS
# ==========================================

w_view_debug = widgets.Checkbox(value=False, description='🛠️ VIEW DEBUG DoG MASK', style={'description_width': 'initial'})
w_use_fill = widgets.Checkbox(value=True, description='1. Use Base Fill')
w_use_dog = widgets.Checkbox(value=True, description='2. Use DoG Edge Mask')
w_use_sobel = widgets.Checkbox(value=True, description='3. Use Sobel Edges')
w_use_bilat = widgets.Checkbox(value=True, description='4. Use Bilateral Blur')
w_strict = widgets.Checkbox(value=True, description='5. Strict Edge Logic')

w_bil_d = widgets.IntSlider(value=9, min=1, max=20, description='Bilat D', continuous_update=False)
w_bil_sig = widgets.IntSlider(value=75, min=10, max=150, description='Bilat Sig', continuous_update=False)

w_dog_sig = widgets.FloatSlider(value=1.2, min=0.1, max=5.0, step=0.1, description='DoG Sigma', continuous_update=False)
w_dog_scale = widgets.FloatSlider(value=1.6, min=1.0, max=3.0, step=0.1, description='DoG Scale', continuous_update=False)
w_dog_tau = widgets.FloatSlider(value=0.99, min=0.5, max=1.2, step=0.01, description='DoG Tau', continuous_update=False)
w_dog_thresh = widgets.FloatSlider(value=0.01, min=0.001, max=0.1, step=0.001, readout_format='.3f', description='DoG Thresh', continuous_update=False)

w_edge_thresh = widgets.IntSlider(value=8, min=0, max=64, description='Edge Thresh', continuous_update=False)

btn_random = widgets.Button(description="🎲 Load Random Image", button_style='success')
btn_random.on_click(load_new_random_image)

ui_toggles = widgets.VBox([w_view_debug, w_use_fill, w_use_dog, w_use_sobel, w_use_bilat, w_strict])
ui_bilat = widgets.VBox([widgets.Label("Skin Flattening:"), w_bil_d, w_bil_sig])
ui_dog = widgets.VBox([widgets.Label("Edge Thickness & Falloff:"), w_dog_sig, w_dog_scale, w_dog_tau, w_dog_thresh])
ui_compute = widgets.VBox([widgets.Label("Compute Logic Strictness:"), w_edge_thresh, btn_random])

ui_panel = widgets.HBox([ui_toggles, ui_bilat, ui_dog, ui_compute])

interactive_plot = widgets.interactive_output(render_pipeline, {
    'view_debug': w_view_debug, 'use_fill': w_use_fill, 'use_dog': w_use_dog, 'use_sobel': w_use_sobel,
    'use_bilateral': w_use_bilat, 'strict_logic': w_strict,
    'bil_d': w_bil_d, 'bil_sig': w_bil_sig,
    'dog_sig': w_dog_sig, 'dog_scale': w_dog_scale, 'dog_tau': w_dog_tau, 'dog_thresh': w_dog_thresh,
    'edge_thresh': w_edge_thresh
})

def update_plot():
    render_pipeline(
        w_view_debug.value, w_use_fill.value, w_use_dog.value, w_use_sobel.value, w_use_bilat.value, w_strict.value,
        w_bil_d.value, w_bil_sig.value, w_dog_sig.value, w_dog_scale.value, w_dog_tau.value, w_dog_thresh.value, w_edge_thresh.value
    )

load_new_random_image()
display(ui_panel, out)

Output()